# 122: SCOPe40 Subdomain Detection Benchmark

**Goal**: Benchmark Kmerseek's ability to detect **subdomain-level relationships** in multidomain proteins — i.e., two proteins that share one domain (same SCOPe superfamily) but have different overall domain architectures.

## Approach

SCOPe40 domain sequences are already individual domains. Each domain comes from a PDB chain, and that chain may contain one or multiple domains.

**Subdomain match (Tier 2 TP)**:
- Query domain D1 and target D2 are from the **same superfamily**
- Their parent PDB chains have **different domain architectures** (different sets of superfamilies)
- BHF-like case: shared functional domain in different protein contexts

**Full homolog (Tier 1 TP / control)**:
- Same superfamily AND same chain architecture — standard full-length homology

**No match (TN)**:
- Different superfamilies

## Data
- SCOPe 2.08 at 40% sequence identity (15,177 domains, 12,748 chains)
- Kmerseek HP k=24 all-vs-all results (`scope_eval.hp.k24.parquet`, 84,715 non-zero-overlap pairs)
- 1,402 PDB chains have ≥2 domains from different superfamilies

## Caveats
- The parquet contains only pairs with **non-zero k-mer overlap**; TN pairs with zero overlap are absent.
- Kmerseek operates on individual SCOPe *domain* sequences, not full-chain sequences. A stricter Tier 2 test would use full multidomain chains as queries.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)

# Paths
SCOPE40_FA  = Path('/Users/olga/data/scope/astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa')
K24_PARQUET = Path('/Users/olga/data/scope/results-scope-pvalue-benchmark/scope_eval.hp.k24.parquet')
RESULTS_DIR = Path('/Users/olga/code/2024-kmerseek-analysis/results')
FIGURES_DIR = Path('/Users/olga/code/2024-kmerseek-analysis/figures')

# Score columns: name → {ascending (True = lower is better), display label}
SCORE_COLS = {
    'poisson_pvalue':              dict(ascending=True,  label='Poisson p-value'),
    'enrichment':                  dict(ascending=False, label='Enrichment'),
    'containment_target_in_query': dict(ascending=False, label='Containment (t-in-q)'),
    'max_containment':             dict(ascending=False, label='Max containment'),
    'jaccard':                     dict(ascending=False, label='Jaccard'),
    'query_tfidf':                 dict(ascending=False, label='TF-IDF'),
    # composite added after data load
}

PALETTE = {
    'no_match':        '#95a5a6',
    'subdomain_match': '#e74c3c',
    'full_homolog':    '#2ecc71',
}
ORDER = ['no_match', 'subdomain_match', 'full_homolog']

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Imports OK.')

## 1. Parse SCOPe40 FASTA → identify multidomain protein chains

In [ ]:
def parse_scope_fasta(fasta_path: Path) -> pl.DataFrame:
    """
    Parse SCOPe FASTA headers into a domain metadata table.

    Header format::
        >d2i3da1 c.69.1.36 (A:2-219) Hypothetical protein {Agrobacterium tumefaciens}

    Domain ID encoding:  d + pdb(4) + chain(1) + domain_num(1)
        d2i3da1  → pdb=2i3d, chain=a, domain 1  (multidomain protein)
        d3m5ba_  → pdb=3m5b, chain=a, underscore (single-domain protein)
    """
    records = []
    with open(fasta_path) as f:
        for line in f:
            if not line.startswith('>'):
                continue
            parts = line[1:].strip().split()
            if len(parts) < 2:
                continue
            dom_id  = parts[0]   # e.g. 'd2i3da1'
            lineage = parts[1]   # e.g. 'c.69.1.36'
            lp = lineage.split('.')
            records.append({
                'dom_id':     dom_id,
                'pdb_chain':  dom_id[1:6],
                'domain_num': dom_id[6] if len(dom_id) > 6 else '_',
                'lineage':    lineage,
                'scop_class': lp[0]            if len(lp) >= 1 else None,
                'scop_fold':  '.'.join(lp[:2]) if len(lp) >= 2 else None,
                'scop_sf':    '.'.join(lp[:3]) if len(lp) >= 3 else None,
                'scop_fam':   lineage          if len(lp) >= 4 else None,
            })

    df = pl.DataFrame(records)
    print(f'Parsed {len(df):,} domains from {fasta_path.name}')
    return df


domains_df = parse_scope_fasta(SCOPE40_FA)
domains_df.head(5)

In [ ]:
# Build domain architecture table: pdb_chain → frozenset(superfamilies)
# Also build per-domain lookup used for pair classification.

chain_to_sfs: dict[str, set] = defaultdict(set)
for row in domains_df.iter_rows(named=True):
    if row['scop_sf'] is not None:
        chain_to_sfs[row['pdb_chain']].add(row['scop_sf'])

domain_lookup: dict[str, dict] = {}
for row in domains_df.iter_rows(named=True):
    pdb_chain = row['pdb_chain']
    arch_key  = frozenset(chain_to_sfs[pdb_chain])
    domain_lookup[row['dom_id']] = {
        'pdb_chain': pdb_chain,
        'arch_key':  arch_key,
        'n_sfs':     len(arch_key),
    }

# Summary stats on chain architectures
chain_sf_counts = pl.Series([len(v) for v in chain_to_sfs.values()])
n_total  = len(chain_to_sfs)
n_multi  = (chain_sf_counts >= 2).sum()
n_single = (chain_sf_counts == 1).sum()

print(f'Total PDB chains:                    {n_total:,}')
print(f'Single-superfamily chains (simple):  {n_single:,}  ({100*n_single/n_total:.1f}%)')
print(f'Multi-superfamily chains (≥2 SFs):   {n_multi:,}  ({100*n_multi/n_total:.1f}%)')
print()

# Show most complex chains
top_chains = (
    domains_df
    .group_by('pdb_chain')
    .agg([
        pl.col('dom_id').alias('domains'),
        pl.col('scop_sf').alias('superfamilies'),
    ])
    .with_columns(pl.col('superfamilies').list.n_unique().alias('n_sf'))
    .sort('n_sf', descending=True)
    .head(6)
)
print('Most complex chains (most distinct superfamilies):')
print(top_chains)

## 2. Load Kmerseek k=24 results and classify pairs

In [ ]:
df_raw = pl.read_parquet(K24_PARQUET)
print(f'Loaded {len(df_raw):,} pairs  ×  {df_raw.width} columns')
print()
print('same_superfamily distribution:')
print(df_raw['same_superfamily'].value_counts())
df_raw.head(3)

In [ ]:
# Extract bare domain ID (first token of name, e.g. 'd2i3da1')
# and add composite score: enrichment × containment_target_in_query / mean_matched_kmer_freq
df = df_raw.with_columns([
    pl.col('query_name').str.split(' ').list.get(0).alias('q_dom_id'),
    pl.col('target_name').str.split(' ').list.get(0).alias('t_dom_id'),
    (
        pl.col('enrichment') *
        pl.col('containment_target_in_query') /
        pl.col('mean_matched_kmer_freq').clip(lower_bound=1e-6)
    ).alias('composite_score'),
])
SCORE_COLS['composite_score'] = dict(ascending=False, label='Composite (enrich×contain/freq)')

print('Sample domain IDs:')
print(df.select(['q_dom_id', 't_dom_id', 'composite_score']).head(4))

In [ ]:
def classify_pair(q_dom_id: str, t_dom_id: str, same_sf: bool) -> str:
    """
    Classify a domain pair:
    - 'subdomain_match': same superfamily, DIFFERENT chain architectures  (Tier 2 TP)
    - 'full_homolog':    same superfamily, IDENTICAL chain architectures   (Tier 1 TP)
    - 'no_match':        different superfamilies                           (TN)
    - 'unknown':         domain not found in FASTA lookup
    """
    if not same_sf:
        return 'no_match'
    q_info = domain_lookup.get(q_dom_id)
    t_info = domain_lookup.get(t_dom_id)
    if q_info is None or t_info is None:
        return 'unknown'
    if q_info['arch_key'] == t_info['arch_key']:
        return 'full_homolog'
    return 'subdomain_match'


print(f'Classifying {len(df):,} pairs...')
pair_types = [
    classify_pair(q, t, sf)
    for q, t, sf in zip(
        df['q_dom_id'].to_list(),
        df['t_dom_id'].to_list(),
        df['same_superfamily'].to_list(),
    )
]
df = df.with_columns(pl.Series('pair_type', pair_types))

counts = df['pair_type'].value_counts().sort('count', descending=True)
print('\nPair type distribution:')
print(counts)

n_sub = counts.filter(pl.col('pair_type') == 'subdomain_match')['count'][0]
n_fh  = counts.filter(pl.col('pair_type') == 'full_homolog')['count'][0]
print(f'\nOf same-superfamily pairs: {100*n_sub/(n_sub+n_fh):.1f}% are subdomain_match (cross-architecture)')

In [ ]:
# Working subsets for evaluation
df_plot = df.filter(pl.col('pair_type') != 'unknown')

# Tier 2: subdomain_match (pos) vs no_match (neg)
df_tier2 = (
    df_plot
    .filter(pl.col('pair_type').is_in(['subdomain_match', 'no_match']))
    .with_columns((pl.col('pair_type') == 'subdomain_match').cast(pl.Int32).alias('label'))
)

# Tier 1: full_homolog (pos) vs no_match (neg)
df_tier1 = (
    df_plot
    .filter(pl.col('pair_type').is_in(['full_homolog', 'no_match']))
    .with_columns((pl.col('pair_type') == 'full_homolog').cast(pl.Int32).alias('label'))
)

print(f'Tier 2 — {df_tier2["label"].sum():,} pos (subdomain_match),  '
      f'{(df_tier2["label"]==0).sum():,} neg')
print(f'Tier 1 — {df_tier1["label"].sum():,} pos (full_homolog),      '
      f'{(df_tier1["label"]==0).sum():,} neg')

## 3. Score distributions by pair type

In [ ]:
score_display = [
    ('enrichment',               'Enrichment',            False),
    ('containment_target_in_query', 'Containment (t-in-q)', False),
    ('poisson_pvalue',           'Poisson p-value',        True),   # log10
    ('jaccard',                  'Jaccard',                False),
    ('query_tfidf',              'TF-IDF score',           False),
    ('composite_score',          'Composite score',        False),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, (col, label, log_scale) in zip(axes, score_display):
    df_pd = df_plot.select(['pair_type', col]).to_pandas().dropna(subset=[col])
    if log_scale:
        df_pd[col] = np.log10(df_pd[col].clip(lower=1e-300))
        label = f'log₁₀({label})'
    sns.violinplot(
        data=df_pd, x='pair_type', y=col,
        order=ORDER, palette=PALETTE,
        ax=ax, inner='quartile', cut=0,
    )
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

fig.suptitle(
    'Score distributions by pair type  (HP k=24, SCOPe40, non-zero overlap pairs)',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '122_score_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved 122_score_distributions.png')

## 4. ROC and PR curves

**Positive = subdomain_match** (Tier 2): can Kmerseek rank cross-architecture domain matches above no_match?

Comparison panel shows **Tier 1** (full_homolog vs no_match) as the upper bound on performance.

In [ ]:
def eval_scores(df_eval: pl.DataFrame) -> pl.DataFrame:
    """Compute ROC-AUC and Average Precision for every scoring column."""
    rows = []
    for col, meta in SCORE_COLS.items():
        if col not in df_eval.columns:
            continue
        sub = df_eval.select(['label', col]).drop_nulls().to_pandas()
        if sub['label'].nunique() < 2:
            continue
        y_true = sub['label'].values
        scores = sub[col].values
        if meta['ascending']:   # lower is better → negate
            scores = -scores
        rows.append({
            'score':   col,
            'label':   meta['label'],
            'ROC_AUC': roc_auc_score(y_true, scores),
            'AP':      average_precision_score(y_true, scores),
        })
    return pl.DataFrame(rows)


t2_metrics = eval_scores(df_tier2)
t1_metrics = eval_scores(df_tier1)

print('=== Tier 2 (subdomain_match vs no_match) ===')
print(t2_metrics.sort('ROC_AUC', descending=True))
print()
print('=== Tier 1 (full_homolog vs no_match) ===')
print(t1_metrics.sort('ROC_AUC', descending=True))

In [ ]:
def plot_roc_pr(df_eval: pl.DataFrame, title: str, axes):
    ax_roc, ax_pr = axes
    cmap  = plt.cm.tab10
    n_pos = int(df_eval['label'].sum())
    n_all = len(df_eval)

    for i, (col, meta) in enumerate(SCORE_COLS.items()):
        if col not in df_eval.columns:
            continue
        sub = df_eval.select(['label', col]).drop_nulls().to_pandas()
        if sub['label'].nunique() < 2:
            continue
        y_true = sub['label'].values
        scores = sub[col].values
        if meta['ascending']:
            scores = -scores

        fpr, tpr, _  = roc_curve(y_true, scores)
        prec, rec, _ = precision_recall_curve(y_true, scores)
        roc_auc      = roc_auc_score(y_true, scores)
        ap           = average_precision_score(y_true, scores)

        ax_roc.plot(fpr, tpr,  color=cmap(i), lw=1.5, label=f'{meta["label"]} ({roc_auc:.3f})')
        ax_pr.plot( rec, prec, color=cmap(i), lw=1.5, label=f'{meta["label"]} ({ap:.3f})')

    ax_roc.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random')
    ax_pr.axhline(n_pos / n_all, color='k', ls='--', lw=0.8,
                  label=f'Random (pos={n_pos/n_all:.3f})')

    for ax, (xl, yl, ttl) in zip(
        [ax_roc, ax_pr],
        [('FPR', 'TPR', f'ROC — {title}'),
         ('Recall', 'Precision', f'PR — {title}')]
    ):
        ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(ttl, fontsize=10)
        ax.legend(fontsize=6, loc='best')


fig, axes = plt.subplots(2, 2, figsize=(14, 11))
plot_roc_pr(df_tier2, 'Tier 2: subdomain_match vs no_match', axes[0])
plot_roc_pr(df_tier1, 'Tier 1: full_homolog vs no_match',    axes[1])

fig.suptitle('Kmerseek HP k=24: ROC and PR curves  (SCOPe40)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '122_roc_pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved 122_roc_pr_curves.png')

## 5. Analysis by domain characteristics

Break down Tier 2 detection by:
- SCOPe structural class of the query domain
- Number of distinct superfamilies in the query protein chain

In [ ]:
CLASS_LABELS = {
    'a': 'All-alpha',
    'b': 'All-beta',
    'c': 'Alpha/beta',
    'd': 'Alpha+beta',
    'e': 'Multi-domain',
    'f': 'Membrane',
    'g': 'Small proteins',
}

# Threshold: 90th percentile of enrichment among no_match (≈ 10% FPR)
thresh_enrich = (
    df_plot.filter(pl.col('pair_type') == 'no_match')['enrichment']
    .quantile(0.90)
)
print(f'Enrichment threshold (90th pct of no_match): {thresh_enrich:.4f}')

# Attach query chain info to Tier 2 pairs
q_n_sfs = [domain_lookup.get(q, {}).get('n_sfs', 1) for q in df_tier2['q_dom_id'].to_list()]
df_t2e = df_tier2.with_columns([
    pl.Series('q_n_sfs', q_n_sfs),
    pl.col('q_scop_class').replace(CLASS_LABELS).alias('structural_class'),
])

# By structural class
class_det = (
    df_t2e
    .group_by('structural_class')
    .agg([
        pl.len().alias('n_pairs'),
        (pl.col('enrichment') > thresh_enrich).sum().alias('n_detected'),
        pl.col('enrichment').median().alias('median_enrichment'),
        pl.col('containment_target_in_query').median().alias('median_containment'),
    ])
    .with_columns((pl.col('n_detected') / pl.col('n_pairs')).alias('detection_rate'))
    .sort('detection_rate', descending=True)
)
print('\n=== Tier 2 detection rate by structural class ===')
print(class_det)

# By number of superfamilies in query chain
arch_det = (
    df_t2e
    .group_by('q_n_sfs')
    .agg([
        pl.len().alias('n_pairs'),
        (pl.col('enrichment') > thresh_enrich).sum().alias('n_detected'),
        pl.col('enrichment').median().alias('median_enrichment'),
        pl.col('containment_target_in_query').median().alias('median_containment'),
    ])
    .with_columns((pl.col('n_detected') / pl.col('n_pairs')).alias('detection_rate'))
    .sort('q_n_sfs')
)
print('\n=== Tier 2 detection rate by # superfamilies in query chain ===')
print(arch_det)

## 6. Visualizations

In [ ]:
# Heatmap: detection rate by structural class × n_sfs in query chain
hm_data = (
    df_t2e
    .group_by(['structural_class', 'q_n_sfs'])
    .agg([
        pl.len().alias('n'),
        (pl.col('enrichment') > thresh_enrich).sum().alias('n_det'),
    ])
    .filter(pl.col('n') >= 3)
    .with_columns((pl.col('n_det') / pl.col('n')).alias('det_rate'))
    .to_pandas()
)

pivot = hm_data.pivot_table(
    index='structural_class', columns='q_n_sfs',
    values='det_rate', aggfunc='mean',
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    pivot, annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Detection rate'},
)
ax.set_title(
    f'Subdomain detection rate  (enrichment > {thresh_enrich:.3f}, HP k=24)\n'
    'structural class of query domain  ×  # unique superfamilies in query chain',
    fontsize=10,
)
ax.set_xlabel('# unique superfamilies in query chain')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '122_detection_rate_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved 122_detection_rate_heatmap.png')

In [ ]:
# Median score bar chart: no_match vs subdomain_match vs full_homolog
metrics_bar = [
    ('enrichment',               'Enrichment'),
    ('containment_target_in_query', 'Containment (t-in-q)'),
    ('jaccard',                  'Jaccard'),
    ('query_tfidf',              'TF-IDF'),
]
medians = {
    pt: {col: df_plot.filter(pl.col('pair_type') == pt)[col].median()
         for col, _ in metrics_bar}
    for pt in ORDER
}

x = np.arange(len(metrics_bar))
w = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for i, pt in enumerate(ORDER):
    vals = [medians[pt][col] for col, _ in metrics_bar]
    ax.bar(x + i*w, vals, w, label=pt, color=PALETTE[pt], alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels([lbl for _, lbl in metrics_bar], rotation=10)
ax.set_ylabel('Median score')
ax.set_title('Median scores by pair type  (HP k=24, SCOPe40)', fontsize=11)
ax.legend(title='Pair type')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '122_median_scores.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved 122_median_scores.png')

## 7. Top subdomain_match hits — novel cross-architecture connections

High-scoring subdomain_match pairs where Kmerseek found a shared domain between proteins with different overall architectures. These represent putative "cryptic homolog" detections that would be missed by full-length sequence comparison.

In [ ]:
top_hits = (
    df_plot
    .filter(pl.col('pair_type') == 'subdomain_match')
    .sort('enrichment', descending=True)
    .head(25)
    .select([
        'q_dom_id', 'q_scop_superfamily',
        't_dom_id', 't_scop_superfamily',
        'enrichment', 'containment_target_in_query',
        'poisson_pvalue', 'jaccard', 'query_tfidf',
    ])
)
print('Top 25 subdomain_match pairs by enrichment:')
print(top_hits)

In [ ]:
# Show domain architecture context: what are the unique/shared SFs for top hits?
print('=== Architecture context for top 10 subdomain matches ===\n')
for row in top_hits.head(10).iter_rows(named=True):
    q_arch = domain_lookup.get(row['q_dom_id'], {}).get('arch_key', set())
    t_arch = domain_lookup.get(row['t_dom_id'], {}).get('arch_key', set())
    shared = sorted(q_arch & t_arch)
    q_only = sorted(q_arch - t_arch)
    t_only = sorted(t_arch - q_arch)
    print(f"  Q {row['q_dom_id']}  T {row['t_dom_id']}")
    print(f"    shared SF : {shared}")
    print(f"    Q-only SF : {q_only}")
    print(f"    T-only SF : {t_only}")
    print(f"    enrichment={row['enrichment']:.3f}  "
          f"containment={row['containment_target_in_query']:.3f}  "
          f"p={row['poisson_pvalue']:.2e}")
    print()

In [ ]:
# Save classified pairs for downstream analyses
out = RESULTS_DIR / '122_subdomain_benchmark_pairs.parquet'
df.select([
    'q_dom_id', 't_dom_id', 'pair_type',
    'q_scop_superfamily', 't_scop_superfamily',
    'q_scop_class', 't_scop_class',
    'enrichment', 'containment_target_in_query',
    'max_containment', 'jaccard', 'query_tfidf',
    'poisson_pvalue', 'composite_score',
    'same_superfamily', 'same_family',
]).write_parquet(out)
print(f'Saved {len(df):,} classified pairs to {out}')